In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 安装必要的库

In [2]:
!pip install ncps

In [3]:
#!pip install dopamine-rl==4.0.0

In [4]:
!pip install "ray[rllib]==2.47.1"

  Using cached gymnasium-1.0.0-py3-none-any.whl.metadata (9.5 kB)
Using cached gymnasium-1.0.0-py3-none-any.whl (958 kB)
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 0.29.0
    Uninstalling gymnasium-0.29.0:
      Successfully uninstalled gymnasium-0.29.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kaggle-environments 1.16.11 requires gymnasium==0.29.0, but you have gymnasium 1.0.0 which is incompatible.
stable-baselines3 2.1.0 requires gymnasium<0.30,>=0.28.1, but you have gymnasium 1.0.0 which is incompatible.


In [32]:
!pip install "ray[rllib]==2.37.0" "gymnasium[mujoco]==0.28.1"

  Using cached ray-2.37.0-cp311-cp311-manylinux2014_x86_64.whl.metadata (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.7/65.7 MB 26.9 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: ray
    Found existing installation: ray 2.30.0
    Uninstalling ray-2.30.0:
      Successfully uninstalled ray-2.30.0


# 导入必要的库文件

In [33]:
import gymnasium
from gymnasium import spaces
import ray
from ray.tune.registry import register_env
from ray.rllib.models import ModelCatalog
from ray.rllib.algorithms.ppo import PPO
import time
import numpy as np
from ray.rllib.models.torch.torch_modelv2 import TorchModelV2
from ray.rllib.models.torch.recurrent_net import RecurrentNetwork as TorchRecurrentNetwork
from ray.rllib.utils.annotations import override
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from ncps.torch import CfC
from ncps.wirings import AutoNCP

In [34]:
class PartialObservation(gymnasium.ObservationWrapper):
    def __init__(self, env: gymnasium.Env, obs_indices: list):
        gymnasium.ObservationWrapper.__init__(self, env)

        obsspace = env.observation_space
        self.obs_indices = obs_indices
        self.observation_space = spaces.Box(
            low=np.array([obsspace.low[i] for i in obs_indices]),
            high=np.array([obsspace.high[i] for i in obs_indices]),
            dtype=np.float32,
        )

        self._env = env

    def observation(self, observation):
        filter_observation = self._filter_observation(observation)
        return filter_observation

    def _filter_observation(self, observation):
        observation = np.array([observation[i] for i in self.obs_indices])
        return observation

In [35]:
def make_partial_observation_cheetah():
    return PartialObservation(
        gymnasium.make("HalfCheetah-v4"), [0, 1, 2, 3, 8, 9, 10, 11, 12]
    )


In [36]:
class CustomRNN(TorchRecurrentNetwork):
    """Custom RNN model using PyTorch and CfC"""

    def __init__(
        self,
        obs_space,
        action_space,
        num_outputs,
        model_config,
        name,
        cell_size=64,
    ):
        super(CustomRNN, self).__init__(
            obs_space, action_space, num_outputs, model_config, name
        )
        self.cell_size = cell_size

        # Define input layers
        self.preprocess_layers = nn.Sequential(
            nn.Linear(obs_space.shape[0], 256),
            nn.SiLU(),
            nn.Linear(256, 256),
            nn.SiLU(),
        )

        # Define the CfC layer
        wiring = AutoNCP(cell_size, num_outputs)  # Assuming num_outputs is the number of outputs
        self.rnn = CfC(256, wiring)  # Input size is 256 after preprocessing

        self.logits = nn.Linear(cell_size, num_outputs)
        self.values = nn.Linear(cell_size, 1)

    @override(TorchRecurrentNetwork)
    def forward_rnn(self, inputs, state, seq_lens):
        # Preprocess the inputs
        preprocessed_inputs = self.preprocess_layers(inputs)

        # Pass through the CfC layer
        rnn_out, state_h = self.rnn(preprocessed_inputs, state[0])

        # Compute logits and values
        logits = self.logits(rnn_out)
        values = self.values(rnn_out)

        return logits, [state_h]

    @override(TorchModelV2)
    def get_initial_state(self):
        return [torch.zeros(self.cell_size, dtype=torch.float32)]

    @override(TorchModelV2)
    def value_function(self):
        return self._value_out


In [37]:
def run_closed_loop(
    algo, rnn_cell_size, n_episodes=10, pertubation_level=0.0, apply_filter=True
):
    env = make_partial_observation_cheetah()
    init_state = None
    state = None
    if rnn_cell_size is not None:
        state = init_state = [torch.zeros(rnn_cell_size, dtype=torch.float32)]
    obs, info = env.reset()
    ep = 0
    ep_rewards = []
    reward = 0
    while ep < n_episodes:
        if pertubation_level > 0.0:
            obs = obs + np.random.default_rng().normal(0, pertubation_level, obs.shape)

        if apply_filter:
            filter = algo.workers.local_worker().filters.get("default_policy")
            obs = filter(obs, update=False)

        if rnn_cell_size is None:
            action = algo.compute_single_action(
                obs, explore=False, policy_id="default_policy"
            )
        else:
            action, state, _ = algo.compute_single_action(
                obs, state=state, explore=False, policy_id="default_policy"
            )
        obs, r, terminated, truncated, info = env.step(action)
        reward += r
        if terminated or truncated:
            ep += 1
            obs, info = env.reset()
            state = init_state
            ep_rewards.append(reward)
            reward = 0
    return np.mean(ep_rewards)


In [38]:
def run_algo(model_name, num_iters):
    config = {
        "env": "my_env",
        "gamma": 0.99,
        "num_gpus": 1,
        "num_workers": 4,
        "num_envs_per_runner": 4,
        "lambda": 0.95,
        "kl_coeff": 1.0,
        "num_sgd_iter": 64,
        "lr": 0.0005,
        "vf_loss_coeff": 0.5,
        "clip_param": 0.1,
        "sgd_minibatch_size": 4096,
        "train_batch_size": 65536,
        "grad_clip": 0.5,
        "batch_mode": "truncate_episodes",
        "observation_filter": "MeanStdFilter",
        "framework": "torch",
        "api_stack": {
            "enable_rl_module_and_learner": False,
            "enable_env_runner_and_connector_v2": False,
        },
    }
    rnn_cell_size = None
    if model_name == "cfc_rnn":
        rnn_cell_size = 64
        config["model"] = {
            "vf_share_layers": True,
            "custom_model": "cfc_rnn",
            "custom_model_config": {
                "cell_size": rnn_cell_size,
            },
        }
    elif model_name == "default":
        pass
    else:
        raise ValueError(f"Unknown model type {model_name}")

    algo = PPO(config=config)
    history = {"reward": [], "reward_noise": [], "iteration": []}
    for iteration in range(1, num_iters + 1):
        algo.train()
        if iteration % 10 == 0 or iteration == 1:
            history["iteration"].append(iteration)
            history["reward"].append(run_closed_loop(algo, rnn_cell_size))
            history["reward_noise"].append(
                run_closed_loop(algo, rnn_cell_size, pertubation_level=0.1)
            )
            print(
                f"{model_name} iteration {iteration}: {history['reward'][-1]:0.2f}, with noise: {history['reward_noise'][-1]:0.2f}"
            )
    return history


In [39]:
if __name__ == "__main__":
    # 注册模型和环境
    ModelCatalog.register_custom_model("cfc_rnn", CustomRNN)
    register_env("my_env", lambda env_config: make_partial_observation_cheetah())

    # 初始化 Ray
    ray.init(num_cpus=4, num_gpus=1, ignore_reinit_error=True)

    # 运行 CFC 模型
    cfc_result = run_algo("cfc_rnn", 10)

    # 关闭 Ray
    ray.shutdown()

    # 注册模型和环境
    ModelCatalog.register_custom_model("cfc_rnn", CustomRNN)
    register_env("my_env", lambda env_config: make_partial_observation_cheetah())

    # 初始化 Ray
    ray.init(num_cpus=4, num_gpus=1, ignore_reinit_error=True)

    # 运行默认 MLP 模型
    mlp_result = run_algo("default", 10)

    # 关闭 Ray
    ray.shutdown()

    # 绘制结果
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(
        mlp_result["iteration"], mlp_result["reward"], label="MLP", color="tab:orange"
    )
    ax.plot(
        cfc_result["iteration"], cfc_result["reward"], label="CfC", color="tab:blue"
    )
    ax.plot(
        mlp_result["iteration"],
        mlp_result["reward_noise"],
        label="MLP (noise)",
        color="tab:orange",
        ls="--",
    )
    ax.plot(
        cfc_result["iteration"],
        cfc_result["reward_noise"],
        label="CfC (noise)",
        color="tab:blue",
        ls="--",
    )
    ax.legend(loc="upper left")
    fig.tight_layout()
    plt.savefig("cfc_vs_mlp.png")

2025-07-04 08:19:08,373	INFO worker.py:1747 -- Calling ray.init() again after it has already been called.


AttributeError: module 'gymnasium.envs.registration' has no attribute 'VectorizeMode'